In [1]:
! nvidia-smi

Fri Mar 14 00:39:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 552.22                 Driver Version: 552.22         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:16:00.0 Off |                  Off |
| 30%   32C    P8             16W /  300W |    2438MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from datasets import load_dataset
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
import gc
import time
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig,SFTTrainer
from PIL import Image

In [3]:
from huggingface_hub import HfFolder, whoami

#Replace 'your_token' with your actual Hugging Face token
hf_token = 'YOUR_HF_TOKEN'  #Write token for Granite-Vision-3.2 Model
HfFolder.save_token(hf_token)

user = whoami()
print(f"Logged in as {user['name']}")

Logged in as aadhibest


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
system_message = """You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.

For circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.

Focus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy."""

In [6]:
def load_image(image_path, target_size=(512, 512)):
    with Image.open(image_path) as img:
        img = img.convert("RGB")  # Ensure compatibility
        img = img.resize(target_size, Image.LANCZOS) #High-Quality Downscaling
        return img.copy()

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": load_image(sample["image"]),
                },
                {
                    
                    "type": "text",
                    "text": sample["conversations"][0]["value"],
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": sample["conversations"][1]["value"]
                }
            ],
        },
    ]

In [7]:
raw_datasets = load_dataset("json", data_files="json_data_12_03_2025_final.json")
split_datasets = raw_datasets["train"].train_test_split(test_size=0.2)
train_dataset = split_datasets["train"] 
eval_dataset = split_datasets["test"]

print(f"Training Dataset Size: {len(train_dataset)}") 
print(f"Evaluation Dataset Size: {len(eval_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Training Dataset Size: 3585
Evaluation Dataset Size: 897


In [8]:
train_dataset[0]

{'id': '1022',
 'image': 'circuit_imgs_03/channel_shuffle/P2P_R30_L75_CHARIMP65_PROP0.1_channel_shuffle.png',
 'conversations': [{'from': 'human',
   'value': 'Describe the circuit and its components in detail'},
  {'from': 'gpt',
   'value': 'This point-to-point (P2P) circuit connects two integrated circuits, IC3 and IC4, using a transmission line to enable high-speed signal communication. IC3 acts as the driver with its output pin labeled "AC86_OUT_33_out" at node 3, which is connected to a series resistor R2. The resistor R2, labeled "HS_PART_RES," has a value of 30 Ohms and serves to improve impedance matching and reduce reflections for better signal integrity. Node B connects this resistor to TL1, a microstrip transmission line designed with a length of 75 mm, a propagation delay of 0.1 mm/ps, and an impedance of 65 Ohms. These parameters ensure controlled signal transmission along the path with minimal distortion or loss. The other end of TL1 terminates at node A, which links to 

In [9]:
train_dataset_formatted = [format_data(sample) for sample in train_dataset]
eval_dataset_formatted = [format_data(sample) for sample in eval_dataset]

In [10]:
train_dataset_formatted[0]

[{'role': 'system',
  'content': [{'type': 'text',
    'text': 'You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.\n\nFor circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.\n\nFocus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy.'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.Image.Image image mode=RGB size=512x512>},
   {'type': 'text',
    'text

In [11]:
# Save train_dataset and eval_dataset to disk
train_dataset_path = "./datasets/train_dataset_13_03_2025_granite"
eval_dataset_path = "./datasets/eval_dataset_13_03_2025_granite"

train_dataset.save_to_disk(train_dataset_path)
eval_dataset.save_to_disk(eval_dataset_path)

print(f"Train dataset saved to: {train_dataset_path}")
print(f"Eval dataset saved to: {eval_dataset_path}")

Saving the dataset (0/1 shards):   0%|          | 0/3585 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/897 [00:00<?, ? examples/s]

Train dataset saved to: ./datasets/train_dataset_13_03_2025_granite
Eval dataset saved to: ./datasets/eval_dataset_13_03_2025_granite


In [12]:
#Remove Model and Clean GPU

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory() #Clearing variable, model and GPU

GPU allocated memory: 0.00 GB
GPU reserved memory: 0.00 GB


In [13]:
model_id = "ibm-granite/granite-vision-3.2-2b"

In [14]:
# BitsAndBytesConfig int-4 config

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["vision_tower", "lm_head"],  # Skip problematic modules
        #llm_int8_enable_fp32_cpu_offload=True
    )
# Load model and tokenizer
model = AutoModelForVision2Seq.from_pretrained(
                                model_id,
                                device_map="auto",
                                torch_dtype=torch.bfloat16,
                                quantization_config=bnb_config,
        ).to("cuda")

processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [15]:
# x = [name for name, _ in model.named_modules() if 'language_model' in name and '_proj' in name]
# x

In [16]:
# Configure LoRA

# peft_config = LoraConfig(
#         r=8,
#         lora_alpha=8,
#         lora_dropout=0.1,
#         target_modules=[name for name, _ in model.named_modules() if 'language_model' in name and '_proj' in name],
#         use_dora=True,
#         init_lora_weights="gaussian"
#     )

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model.add_adapter(peft_config)
model.enable_adapters()
model = get_peft_model(model, peft_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 3,125,248 || all params: 2,978,522,176 || trainable%: 0.1049


In [17]:
# Configure training arguments using SFTConfig

training_args = SFTConfig(
    output_dir="granite-vision-3.2-2b-13-03-2025", 
    logging_dir="granite-vision-3.2-2b-13-03-2025/logs",
    num_train_epochs=1,  
    per_device_train_batch_size=4,  
    gradient_accumulation_steps=4, 
    remove_unused_columns = False,
    optim="adamw_torch_fused",  
    learning_rate=2e-4,  
    lr_scheduler_type="constant",
    logging_steps=10, 
    save_strategy="steps", 
    save_steps=25,  
    save_total_limit=10,
    metric_for_best_model="eval_loss",  
    greater_is_better=False, 
    bf16=True,  
    max_grad_norm=0.3,  
    warmup_ratio=0.03, 
    push_to_hub=True, 
    report_to="tensorboard",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="",  
    dataset_kwargs={"skip_prepare_dataset": True},
)

In [18]:
def collate_fn(examples):
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]

    image_inputs = []
    for example in examples:
      image = example[1]['content'][0]['image']
      if image.mode != 'RGB':
          image = image.convert('RGB')
      image_inputs.append([image])

    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    assistant_tokens = processor.tokenizer("<|assistant|>", return_tensors="pt")['input_ids'][0]
    eos_token = processor.tokenizer("<|end_of_text|>", return_tensors="pt")['input_ids'][0]

    for i in range(batch["input_ids"].shape[0]):
        apply_loss = False
        for j in range(batch["input_ids"].shape[1]):
            if not apply_loss:
                labels[i][j] = -100
            if ((j>=len(assistant_tokens)+1) and
                torch.all(batch["input_ids"][i][j+1-len(assistant_tokens):j+1]==assistant_tokens)):
                apply_loss = True
            if batch["input_ids"][i][j]==eos_token:
                apply_loss = False

    batch["labels"] = labels

    return batch

In [19]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

C:\Users\hal-pg2425-5\AppData\Local\Temp\8\ipykernel_34396\2552207196.py:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [20]:
%load_ext tensorboard
%tensorboard --logdir granite-vision-3.2-2b-13-03-2025/logs --port=6007


In [21]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,7.125400
20,6.443100
30,5.885900
40,5.342800
50,5.038900
60,4.844700
70,4.257800
80,3.986900
90,3.930800
100,3.563200


TrainOutput(global_step=224, training_loss=4.07896625995636, metrics={'train_runtime': 9844.5068, 'train_samples_per_second': 0.364, 'train_steps_per_second': 0.023, 'total_flos': 2.6584346966898432e+17, 'train_loss': 4.07896625995636})

In [22]:
trainer.save_model(training_args.output_dir)

In [23]:
trainer.push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/aadhibest/granite-vision-3.2-2b-13-03-2025/commit/1bfa3cf3248fed31b64491ae4714cb35ec3e30b6', commit_message='End of training', commit_description='', oid='1bfa3cf3248fed31b64491ae4714cb35ec3e30b6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aadhibest/granite-vision-3.2-2b-13-03-2025', endpoint='https://huggingface.co', repo_type='model', repo_id='aadhibest/granite-vision-3.2-2b-13-03-2025'), pr_revision=None, pr_num=None)

In [24]:
clear_memory()

GPU allocated memory: 0.02 GB
GPU reserved memory: 0.08 GB
